# 02 Exploratory Data Analysis

Notebook ini berisi analisis produksi, kualitas, downtime, dan biaya sebagai dasar dashboard.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'manufacturing_clean.csv', parse_dates=['production_date'])
df.head()

In [ ]:
monthly = df.groupby('year_month').agg(
    planned_quantity=('planned_quantity', 'sum'),
    produced_quantity=('produced_quantity', 'sum'),
    defect_quantity=('defect_quantity', 'sum'),
    downtime_minutes=('downtime_minutes', 'sum'),
    production_cost=('production_cost', 'sum')
)
monthly['achievement_pct'] = monthly['produced_quantity'] / monthly['planned_quantity'] * 100
monthly['defect_rate_pct'] = monthly['defect_quantity'] / monthly['produced_quantity'] * 100
monthly.tail(12)

In [ ]:
df.groupby('product_line').agg(
    produced_quantity=('produced_quantity', 'sum'),
    defect_quantity=('defect_quantity', 'sum'),
    downtime_minutes=('downtime_minutes', 'sum'),
    production_cost=('production_cost', 'sum')
).assign(
    defect_rate_pct=lambda x: x['defect_quantity'] / x['produced_quantity'] * 100,
    cost_per_unit=lambda x: x['production_cost'] / x['produced_quantity']
).sort_values('defect_rate_pct', ascending=False)

In [ ]:
df.groupby('machine_id').agg(
    downtime_minutes=('downtime_minutes', 'sum'),
    planned_quantity=('planned_quantity', 'sum'),
    produced_quantity=('produced_quantity', 'sum'),
    defect_quantity=('defect_quantity', 'sum')
).assign(
    achievement_pct=lambda x: x['produced_quantity'] / x['planned_quantity'] * 100,
    defect_rate_pct=lambda x: x['defect_quantity'] / x['produced_quantity'] * 100
).sort_values('downtime_minutes', ascending=False).head(10)